In [1]:
import pyspark
from pyspark.sql import SparkSession

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA, Imputer
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.sql.functions import mean, col, expr
import numpy as np
import time

In [2]:
spark = SparkSession.builder.appName("ce53") \
    .config("SPARK_LOCAL_IP", "192.168.1.2") \
    .master("spark://192.168.1.2:7077") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.maxResultSize", "3g") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "25g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "16") \
    .config("spark.shuffle.partitions", "180") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.sql.execution.arrow.enabled", "true") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/07/14 19:34:13 WARN Utils: Your hostname, ubuntu-virtual-machine resolves to a loopback address: 127.0.1.1; using 192.168.1.106 instead (on interface ens33)
24/07/14 19:34:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/14 19:34:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.stop()

In [4]:
parquet_files = ["hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-12 - 2021-12-19/part-00000-d512890f-d1e9-49d5-a136-f87f0183cb4d-c000.snappy.parquet", 
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-19 - 2021-12-26/part-00000-d28b031b-bff1-4e16-853a-9b7d896627e7-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-26 - 2022-01-02/part-00000-94d13437-ae00-4a8c-9f38-edd0196cfdee-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-01-02 - 2022-01-09/part-00000-745e350a-da9e-4619-bd52-8cc23bb41ad5-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-08-28 - 2022-09-04/part-00000-9a46dd05-4b06-4a39-a45b-5c8460b6c37b-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-04 - 2022-09-11/part-00000-ea53b0e8-d346-44e3-9a87-1f60ac35c610-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-11 - 2022-09-18/part-00000-f9afaec0-242e-41e7-906d-a42681515d75-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-18 - 2022-09-25/part-00000-9ac876be-c07d-4a18-878d-959efa26f484-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-25 - 2022-10-02/part-00000-be6d0798-554d-4c7a-9fef-d4c07aa0ce19-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-02 - 2022-10-09/part-00000-2b76f9cc-0710-45e4-9e33-98ad5808ee79-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-16 - 2022-10-23/part-00000-9aeb279c-81c6-4481-9b30-d35d4d194fea-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-23 - 2022-10-30/part-00000-23fdcfa3-9dd3-4c72-886c-e945bfcf92e1-c000.snappy.parquet",                
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-09 - 2022-10-16/part-00000-b2b625bc-5816-4586-b977-35f9ed4487fd-c000.snappy.parquet"]

In [5]:
#Read the parquet files
df = spark.read.parquet(*parquet_files, inferSchema=True)

24/07/14 19:34:17 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/14 19:34:17 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/14 19:34:17 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


In [6]:
#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

label_counts.show()

+--------------------+------+
|        label_tactic| count|
+--------------------+------+
|          Collection|     1|
| Command and Control|    17|
|   Credential Access|     1|
|     Defense Evasion|  3064|
|           Discovery| 16819|
|           Execution|    30|
|      Initial Access|    19|
|    Lateral Movement|    11|
|         Persistence|    10|
|Privilege Escalation|  3066|
|      Reconnaissance| 51492|
|Resource Development|275471|
|                none|350339|
+--------------------+------+



In [7]:
start_time = time.time()

#Drop labels and get remaining counts
labels_to_drop = ['Exfiltration',
                  'Initial Access',
                  'Lateral Movement', 
                  'Persistence',
                  'Collection',
                  'Command and Control',
                  'Execution',
                  'Credential Access']
#                 'Discovery']
#                 'Defense Evasion'
#                 'Privilege Escalation'
#                 'Resource Development'
#                 'Reconnaissance'

df = df.filter(~col("label_tactic").isin(labels_to_drop))
#filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

#filtered_label_counts.show()

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.07166624069213867 seconds


In [8]:
#Drop uid feature
df = df.drop('uid')
df = df.drop('label_technique')
df = df.drop('label_binary')

In [9]:
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

filtered_label_counts.show()

+--------------------+------+
|        label_tactic| count|
+--------------------+------+
|     Defense Evasion|  3064|
|           Discovery| 16819|
|Privilege Escalation|  3066|
|      Reconnaissance| 51492|
|Resource Development|275471|
|                none|350339|
+--------------------+------+



In [10]:
start_time = time.time()

#Columns to index
columns_to_index = ['service', 
                    'conn_state', 
                    'history', 
                    'proto', 
                    'dest_ip_zeek', 
                    'community_id', 
                    'src_ip_zeek',
                    'datetime',
                    'local_resp',
                    'local_orig',
                    'label_tactic']

#Cast datetime, local_resp, local_orig to String
df = df.withColumn("datetime", col("datetime").cast("string"))
df = df.withColumn("local_resp", col("local_resp").cast("string"))
df = df.withColumn("local_orig", col("local_orig").cast("string"))

#Impute null values with empty string
for column in columns_to_index:
    df = df.fillna('', subset=[column])

In [11]:
#Split the into training and test sets
start_time = time.time()
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.047342777252197266 seconds


In [12]:
#StringIndexer
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").setHandleInvalid("keep") for column in columns_to_index]

#Chain indexers together
pipeline = Pipeline(stages=indexers).fit(train_data)

#Fit and transform the data
train_data_indexed = pipeline.transform(train_data)
test_data_indexed = pipeline.transform(test_data)

#Drop original columns
train_data_indexed = train_data_indexed.drop(*columns_to_index)
train_data_indexed = train_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")
test_data_indexed = test_data_indexed.drop(*columns_to_index)
test_data_indexed = test_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")

#print("train_indexed columns: ", train_data_indexed.columns)
#print("test_indexed columns: ", test_data_indexed.columns)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 38.8300085067749 seconds


In [13]:
start_time = time.time()

#List of numeric column names
numeric_columns = ['resp_pkts', 
                   'orig_ip_bytes', 
                   'missed_bytes', 
                   'duration', 
                   'orig_pkts',
                   'resp_ip_bytes', 
                   'dest_port_zeek', 
                   'orig_bytes', 
                   'resp_bytes',
                   'src_port_zeek', 
                   'ts']

#Create Imputer
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

#Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data_indexed)

#Apply the Imputer to the training data
train_data_imputed = imputer_model.transform(train_data_indexed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Apply the Imputer to the test data
start_time = time.time()
test_data_imputed = imputer_model.transform(test_data_indexed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_data_imputed = train_data_imputed.drop(*numeric_columns)
test_data_imputed = test_data_imputed.drop(*numeric_columns)

Execution time: 2.8662726879119873 seconds
Execution time: 0.024641036987304688 seconds


In [14]:
print("\nTrain data imputed: ", train_data_imputed.columns)
print("\n")
print("Test data imputed: ", test_data_imputed.columns)


Train data imputed:  ['service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic', 'resp_pkts_imputed', 'orig_ip_bytes_imputed', 'missed_bytes_imputed', 'duration_imputed', 'orig_pkts_imputed', 'resp_ip_bytes_imputed', 'dest_port_zeek_imputed', 'orig_bytes_imputed', 'resp_bytes_imputed', 'src_port_zeek_imputed', 'ts_imputed']


Test data imputed:  ['service_indexed', 'conn_state_indexed', 'history_indexed', 'proto_indexed', 'dest_ip_zeek_indexed', 'community_id_indexed', 'src_ip_zeek_indexed', 'datetime_indexed', 'local_resp_indexed', 'local_orig_indexed', 'label_tactic', 'resp_pkts_imputed', 'orig_ip_bytes_imputed', 'missed_bytes_imputed', 'duration_imputed', 'orig_pkts_imputed', 'resp_ip_bytes_imputed', 'dest_port_zeek_imputed', 'orig_bytes_imputed', 'resp_bytes_imputed', 'src_port_zeek_imputed', 'ts_imputed']


In [15]:
start_time = time.time()

#Create VectorAssembler
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed") or column.endswith("_indexed")]
#print("Columns to assemble: ", columns_to_assemble)

assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

#Transform the training data
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Transform the test data
start_time = time.time()
test_data_assembled = assembler.transform(test_data_imputed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Select the features and label columns
train_data_assembled = train_data_assembled.select("features", "label_tactic")
test_data_assembled = test_data_assembled.select("features", "label_tactic")

Execution time: 0.7353684902191162 seconds
Execution time: 0.6462423801422119 seconds


In [16]:
print("Train_data_assembled columns: ", train_data_assembled.columns)
print("Test_data_assembled columns: ", test_data_assembled.columns)

Train_data_assembled columns:  ['features', 'label_tactic']
Test_data_assembled columns:  ['features', 'label_tactic']


In [17]:
from pyspark.ml.feature import StandardScaler


start_time = time.time()

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)

train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/14 19:35:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/07/14 19:35:25 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/14 19:35:30 WARN DAGScheduler: Broadcasting large task binary with size 33.2 MiB


Execution time: 8.774745464324951 seconds


In [18]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic")

In [19]:
test_data_normalized

DataFrame[features_normalized: vector, label_tactic: double]

In [20]:
train_data_normalized

DataFrame[features_normalized: vector, label_tactic: double]

In [21]:
import pandas as pd

start_time = time.time()

# Convert Spark DataFrame to Pandas DataFrame
train_pd = train_data_normalized.toPandas()
test_pd = test_data_normalized.toPandas()

# Extract features and labels from Pandas DataFrames
X_train = train_pd['features_normalized'].values.tolist()
y_train = train_pd['label_tactic'].values.tolist()
X_test = test_pd['features_normalized'].values.tolist()
y_test = test_pd['label_tactic'].values.tolist()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/14 19:35:35 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB
24/07/14 19:35:53 WARN DAGScheduler: Broadcasting large task binary with size 33.3 MiB


Execution time: 27.068443298339844 seconds


# sklearn LDA

In [22]:
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# # Define the number of components for LDA
# n_components = 1 

# start_time = time.time()

# # Perform Linear Discriminant Analysis in scikit-learn with the specified number of components
# lda = LinearDiscriminantAnalysis(n_components=n_components)
# X_train_lda = lda.fit_transform(X_train, y_train)
# X_test_lda = lda.transform(X_test)

# end_time = time.time()
# execution_time = end_time - start_time
# print("Execution time:", execution_time, "seconds")


# start_time = time.time()

# implementation of LDA

In [23]:
import time

In [24]:
# Function to add two vectors, handling None values
def add_vectors(v1, v2):
    result = []
    for a, b in zip(v1, v2):
        if a is None and b is None:
            result.append(None)
        elif a is None:
            result.append(b)
        elif b is None:
            result.append(a)
        else:
            result.append(a + b)
    return result

# Function to divide a vector by a scalar, handling None and str values
def divide_vector(v, divisor):
    result = []
    for a in v:
        if a is None:
            result.append(None)
        elif isinstance(a, (int, float)):
            result.append(a / divisor)
        else:
            result.append(a)
    return result

In [25]:

# Calculate class means
start_time = time.time()

class_sums_counts = {}
for label, features in zip(y_train, X_train):
    if label not in class_sums_counts:
        class_sums_counts[label] = (features, 1)
    else:
        current_sum, count = class_sums_counts[label]
        class_sums_counts[label] = (add_vectors(current_sum, features), count + 1)

class_means = {label: divide_vector(sums_counts[0], sums_counts[1]) for label, sums_counts in class_sums_counts.items()}

# Calculate overall mean
overall_sum = [0] * len(X_train[0])
for features in X_train:
    overall_sum = add_vectors(overall_sum, features)

overall_mean = divide_vector(overall_sum, len(X_train))

In [26]:

# Compute the between-class scatter matrix
start_time = time.time()

def compute_between_class_scatter(class_mean, overall_mean, count):
    diff = [a - b for a, b in zip(class_mean, overall_mean) if isinstance(a, (int, float)) and isinstance(b, (int, float))]
    return [[count * a * b for b in diff] for a in diff]

class_counts = {label: 0 for label in class_means.keys()}
for label in y_train:
    class_counts[label] += 1

between_class_scatter = [[0] * len(overall_mean) for _ in range(len(overall_mean))]

for label, count in class_counts.items():
    class_scatter = compute_between_class_scatter(class_means[label], overall_mean, count)
    between_class_scatter = [[a + b for a, b in zip(row_x, row_y)] for row_x, row_y in zip(between_class_scatter, class_scatter)]

# Compute the within-class scatter matrix
def compute_within_class_scatter(class_mean, point):
    diff = [a - b for a, b in zip(point, class_mean) if isinstance(a, (int, float)) and isinstance(b, (int, float))]
    return [[a * b for b in diff] for a in diff]

within_class_scatter = [[0] * len(overall_mean) for _ in range(len(overall_mean))]

for features, label in zip(X_train, y_train):
    class_scatter = compute_within_class_scatter(class_means[label], features)
    within_class_scatter = [[a + b for a, b in zip(row_x, row_y)] for row_x, row_y in zip(within_class_scatter, class_scatter)]


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 40.19485068321228 seconds


In [27]:
# Compute eigenvalues and eigenvectors for the scatter matrices
start_time = time.time()

def matrix_multiply(A, B):
    return [[sum(a * b for a, b in zip(row_a, col_b)) for col_b in zip(*B)] for row_a in A]

def transpose(matrix):
    return list(map(list, zip(*matrix)))

def inverse(matrix):
    n = len(matrix)
    identity = [[float(i == j) for i in range(n)] for j in range(n)]
    augmented = [row + identity_row for row, identity_row in zip(matrix, identity)]
    for i in range(n):
        pivot = augmented[i][i]
        augmented[i] = [x / pivot for x in augmented[i]]
        for j in range(n):
            if i != j:
                factor = augmented[j][i]
                augmented[j] = [a - factor * b for a, b in zip(augmented[j], augmented[i])]
    inverse_matrix = [row[n:] for row in augmented]
    return inverse_matrix

inv_within_class_scatter = inverse(within_class_scatter)
eig_matrix = matrix_multiply(inv_within_class_scatter, between_class_scatter)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.004748821258544922 seconds


In [28]:
# Use a simple power iteration method to find the eigenvector corresponding to the largest eigenvalue

start_time = time.time()

def power_iteration(matrix, num_simulations: int):
    b_k = [1.0] * len(matrix)
    for _ in range(num_simulations):
        b_k1 = [sum(a * b for a, b in zip(row, b_k)) for row in matrix]
        b_k1_norm = sum(x ** 2 for x in b_k1) ** 0.5
        b_k = [x / b_k1_norm for x in b_k1]
    return b_k

eigenvector = power_iteration(eig_matrix, 100)

# Transform the data to the new feature space
def project_point(point, eigenvector):
    return sum(a * b for a, b in zip(point, eigenvector) if isinstance(a, (int, float)) and isinstance(b, (int, float)))

X_train_lda = [project_point(features, eigenvector) for features in X_train]
X_test_lda = [project_point(features, eigenvector) for features in X_test]

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


Execution time: 8.075006484985352 seconds


In [29]:
n_components = 1

In [30]:
# Convert the transformed arrays back to Pandas DataFrames
start_time = time.time()

train_pd_lda = pd.DataFrame(X_train_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])
test_pd_lda = pd.DataFrame(X_test_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])

# Combine the transformed features with the labels
train_pd_lda['label_tactic'] = y_train
test_pd_lda['label_tactic'] = y_test

# Convert Pandas DataFrames back to Spark DataFrames
train_lda = spark.createDataFrame(train_pd_lda)
test_lda = spark.createDataFrame(test_pd_lda)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 9.96120285987854 seconds


# output from the LDA implementation on pyspark

In [31]:
#Show the adjusted data
train_lda.show()
test_lda.show()

+-------------------+------------+
|      lda_feature_1|label_tactic|
+-------------------+------------+
| -1.024860530418271|         0.0|
|-1.0071392986081444|         0.0|
|-1.0132183562252586|         0.0|
|-1.0299094810778986|         0.0|
|-1.0130717923668333|         0.0|
|-1.0136591214552297|         0.0|
| -1.017138191421646|         0.0|
|-1.0283277332538752|         0.0|
|-1.0326391785503157|         0.0|
|-1.0156372657224533|         0.0|
|-1.0126548253936705|         0.0|
|-1.0359946377422202|         0.0|
|-1.0048391732000805|         0.0|
|-1.0094605994451158|         0.0|
| -1.033330928823245|         0.0|
|-1.0547350567469371|         0.0|
|-1.0230417899109407|         0.0|
|-1.0378050437764033|         0.0|
|-1.0349414925227045|         0.0|
|-1.0037789722002326|         0.0|
+-------------------+------------+
only showing top 20 rows



+-------------------+------------+
|      lda_feature_1|label_tactic|
+-------------------+------------+
|-1.0785721569330586|         0.0|
| -1.079985805564549|         0.0|
|-1.0653849471468346|         0.0|
|-1.0792607802877623|         0.0|
|-1.0898155897310695|         0.0|
|-1.0616850939179503|         0.0|
|-1.0623759365090233|         0.0|
|-1.0588527806175407|         0.0|
| -1.057675962967354|         0.0|
|-1.0753141269332165|         0.0|
|-1.0485519219844772|         0.0|
|-1.0752372519000004|         0.0|
|-1.0616399830923173|         0.0|
|-1.0599224189654002|         0.0|
|-1.0745869809102817|         0.0|
|-1.0593768598761335|         0.0|
|-1.0676589057905201|         0.0|
|-1.0457137165675452|         0.0|
| -1.060398796576811|         0.0|
|-1.0543403407780423|         0.0|
+-------------------+------------+
only showing top 20 rows



# output from sklearn LDA

In [32]:
#Show the adjusted data - sklearn
train_lda.show()
test_lda.show()

+-------------------+------------+
|      lda_feature_1|label_tactic|
+-------------------+------------+
| -1.024860530418271|         0.0|
|-1.0071392986081444|         0.0|
|-1.0132183562252586|         0.0|
|-1.0299094810778986|         0.0|
|-1.0130717923668333|         0.0|
|-1.0136591214552297|         0.0|
| -1.017138191421646|         0.0|
|-1.0283277332538752|         0.0|
|-1.0326391785503157|         0.0|
|-1.0156372657224533|         0.0|
|-1.0126548253936705|         0.0|
|-1.0359946377422202|         0.0|
|-1.0048391732000805|         0.0|
|-1.0094605994451158|         0.0|
| -1.033330928823245|         0.0|
|-1.0547350567469371|         0.0|
|-1.0230417899109407|         0.0|
|-1.0378050437764033|         0.0|
|-1.0349414925227045|         0.0|
|-1.0037789722002326|         0.0|
+-------------------+------------+
only showing top 20 rows

+-------------------+------------+
|      lda_feature_1|label_tactic|
+-------------------+------------+
|-1.0785721569330586|        

In [33]:
start_time = time.time()

# List of columns to assemble
columns_to_assemble = [f'lda_feature_{i+1}' for i in range(n_components)]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform train_lda
train_lda_assembled = assembler.transform(train_lda)

# Transform test_lda
test_lda_assembled = assembler.transform(test_lda)

# Select only the assembled features and label column for both datasets
train_lda_assembled = train_lda_assembled.select("features", "label_tactic")
test_lda_assembled = test_lda_assembled.select("features", "label_tactic")

# Show the schema of the assembled train_lda DataFrame
train_lda_assembled.printSchema()

# Show the schema of the assembled test_lda DataFrame
test_lda_assembled.printSchema()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

root
 |-- features: vector (nullable = true)
 |-- label_tactic: double (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label_tactic: double (nullable = true)

Execution time: 0.14063715934753418 seconds


In [34]:
#Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)
ovr = OneVsRest(classifier=svm, labelCol="label_tactic")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.05425882339477539 seconds


In [35]:
#Fit the model
start_time = time.time()
svm_model = ovr.fit(train_lda_assembled)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/14 19:37:09 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/14 19:37:29 WARN BlockManager: Asked to remove block broadcast_228, which does not exist
24/07/14 19:37:45 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/07/14 19:38:06 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/07/14 19:38:09 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 
24/07/14 19:38:28 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 


Execution time: 97.1978542804718 seconds


In [36]:
#Make predictions
start_time = time.time()

predictions = svm_model.transform(test_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/14 19:38:45 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


Execution time: 0.5427381992340088 seconds


In [37]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

Accuracy: 0.8933317433818269
Precision: 0.809447525163062
Recall: 0.8933317433818269
F1-Score: 0.846375577224376
FPR by Label: 0.0
Weighted FPR: 0.06901166545199512


In [38]:
start_time = time.time()

#Extract predictions and labels
predictions_and_labels = predictions.select("prediction", "label_tactic")

#Calculate false positives and true negatives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic== 0)).count()
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic == 0)).count()

#Calculate FPR
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

False Positive Rate: 9.52998132123661e-06
Execution time: 1.8379967212677002 seconds


In [39]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

/home/ubuntu/.local/lib/python3.10/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Area under ROC =  0.9999952350093394


In [40]:
spark.sparkContext.stop()